In [4]:
# ============================================================================
# ENHANCED TRAINING AND PREDICTION PIPELINE FOR TELECOM CHURN
# ============================================================================
import pandas as pd
import numpy as np
import re
from datetime import datetime
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler, QuantileTransformer
from sklearn.metrics import classification_report, accuracy_score, make_scorer, roc_auc_score, f1_score, confusion_matrix
from sklearn.feature_selection import SelectFromModel, VarianceThreshold, mutual_info_classif
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')


# ============================================================================
# HELPER FUNCTIONS
# ============================================================================
def clean_feature_names(columns):
    """Clean feature names to remove special characters for LightGBM compatibility"""
    cleaned = []
    for col in columns:
        cleaned_name = re.sub(r'[^A-Za-z0-9_]', '_', str(col))
        cleaned_name = re.sub(r'_+', '_', cleaned_name)
        cleaned_name = cleaned_name.strip('_')
        cleaned.append(cleaned_name)
    return cleaned

def create_advanced_features(df, is_train=True, enhancement=True):
    """
    Advanced feature engineering with domain-specific insights
    """
    data = df.copy()
    
    # ========================================================================
    # 1. BASIC RATIO AND INTERACTION FEATURES
    # ========================================================================
    
    # ARPU and billing features
    data['arpu_per_year'] = data['arpu'] / 12
    data['avg_bill_per_mou'] = data['l3m_avg_bill_dura'] / (data['l3m_avg_mou'] + 1)
    data['bill_usage_ratio'] = data['cm_tot_bill_dura'] / (data['l3m_avg_bill_dura'] + 1)
    data['arpu_stability'] = data['arpu'] / (data['l3m_avg_bill_dura'].std() + 1) if is_train else 0
    
    # Data usage features
    data['flux_usage_ratio'] = data['cm_flux_use'] / (data['cm_base_plan_flux'] + data['cm_chos_plan_flux'] + 1)
    data['4g_usage_ratio'] = data['flux_4g_use'] / (data['cm_flux_use'] + 1)
    data['upload_download_ratio'] = data['flux_up_4g_sum'] / (data['flux_down_4g_sum'] + 1)
    data['avg_daily_flux'] = data['cm_flux_use'] / (data['gprs_days'] + 1)
    data['flux_efficiency'] = data['cm_flux_use'] / (data['cm_tot_bill_dura'] + 1)
    
    # ========================================================================
    # 2. TIME-BASED BEHAVIORAL PATTERNS
    # ========================================================================
    
    # Weekday vs Weekend patterns
    data['wday_flux_total'] = data['wday_day_flux'] + data['wday_night_flux']
    data['nwday_flux_total'] = data['nwday_day_flux'] + data['nwday_night_flux']
    data['wday_vs_nwday'] = data['wday_flux_total'] / (data['nwday_flux_total'] + 1)
    data['day_vs_night_wday'] = data['wday_day_flux'] / (data['wday_night_flux'] + 1)
    data['day_vs_night_nwday'] = data['nwday_day_flux'] / (data['nwday_night_flux'] + 1)
    
    # Overall day/night preference
    data['total_day_flux'] = data['wday_day_flux'] + data['nwday_day_flux']
    data['total_night_flux'] = data['wday_night_flux'] + data['nwday_night_flux']
    data['day_night_preference'] = data['total_day_flux'] / (data['total_night_flux'] + 1)
    data['is_night_user'] = (data['total_night_flux'] > data['total_day_flux']).astype(int)
    
    # ========================================================================
    # 3. ADVANCED USAGE PATTERN FEATURES
    # ========================================================================
    
    # Over-plan usage insights
    data['has_over_plan'] = (data['cm_over_plan_flux'] > 0).astype(int)
    data['over_plan_ratio'] = data['cm_over_plan_flux'] / (data['cm_base_plan_flux'] + data['cm_chos_plan_flux'] + 1)
    data['over_plan_severity'] = np.where(
        data['over_plan_ratio'] > 0.5, 2,
        np.where(data['over_plan_ratio'] > 0.2, 1, 0)
    )
    
    # Voice usage patterns
    data['local_voice_ratio'] = data['cm_local_voice_dura'] / (data['cm_tot_bill_dura'] + 1)
    data['voice_data_ratio'] = data['cm_tot_bill_dura'] / (data['cm_flux_use'] + 1)
    data['is_voice_heavy'] = (data['cm_tot_bill_dura'] > data['cm_flux_use']).astype(int)
    
    # ========================================================================
    # 4. BROADBAND AND CONNECTIVITY FEATURES
    # ========================================================================
    
    if 'bd_flux_m' in data.columns:
        data['has_broadband'] = (data['bd_flux_m'] > 0).astype(int)
        data['bd_usage_intensity'] = data['bd_flux_m'] / (data['bd_dur_m'] + 1)
        data['bd_avg_session_dur'] = data['bd_dur_m'] / (data['bd_cnt_m'] + 1)
        data['mobile_bd_ratio'] = data['cm_flux_use'] / (data['bd_flux_m'] + 1)
        data['bd_session_frequency'] = data['bd_cnt_m'] / 30  # Average sessions per day
        data['bd_engagement_score'] = data['bd_flux_m'] * data['bd_dur_m'] / 1e6
    
    # ========================================================================
    # 5. TV AND ENTERTAINMENT ENGAGEMENT
    # ========================================================================
    
    if 'user_duration_m' in data.columns:
        data['tv_engagement'] = data['user_duration_m'] / (data['login_times_m'] + 1)
        data['click_watch_ratio'] = data['click_times_m'] / (data['watch_times_m'] + 1)
        data['active_days_ratio'] = data['open_day_m'] / 30
        data['avg_session_duration'] = data['user_duration_m'] / (data['login_times_m'] + 1)
        data['engagement_intensity'] = data['watch_times_m'] / (data['open_day_m'] + 1)
        data['tv_loyalty'] = data['open_day_m'] * data['login_times_m'] / 900  # Normalized loyalty score
    
    # ========================================================================
    # 6. COMPREHENSIVE VIDEO USAGE FEATURES
    # ========================================================================
    
    video_cols = ['gm_use_dur', 'shrt_vid_use_dur', 'long_vid_use_dur', 
                  'anchor_use_dur', 'wtch_liv_use_dur', 'netdisk_use_dur']
    
    if all(col in data.columns for col in video_cols):
        # Total usage
        data['total_video_dur'] = data[video_cols].sum(axis=1)
        
        # Content type preferences
        data['game_video_ratio'] = data['gm_use_dur'] / (data['total_video_dur'] + 1)
        data['short_long_ratio'] = data['shrt_vid_use_dur'] / (data['long_vid_use_dur'] + 1)
        data['live_ondemand_ratio'] = data['wtch_liv_use_dur'] / (data['long_vid_use_dur'] + data['shrt_vid_use_dur'] + 1)
        
        # Video diversity
        data['video_diversity'] = (data[video_cols] > 0).sum(axis=1)
        
        # Day vs night patterns
        data['game_day_ratio'] = data['gm_dayt_use_dur'] / (data['gm_use_dur'] + 1)
        data['video_night_usage'] = (data['shrt_vid_ngt_use_dur'] + 
                                      data['long_vid_ngt_use_dur'] + 
                                      data['anchor_ngt_use_dur'])
        data['video_day_usage'] = (data['shrt_vid_dayt_use_dur'] + 
                                    data['long_vid_dayt_use_dur'] + 
                                    data['anchor_dayt_use_dur'])
        data['video_day_night_ratio'] = data['video_day_usage'] / (data['video_night_usage'] + 1)
        
        # Content consumption patterns
        data['is_binge_watcher'] = ((data['long_vid_use_dur'] > data['total_video_dur'].quantile(0.75)) & 
                                     (data['long_vid_use_dur'] > 0)).astype(int) if is_train else 0
        data['is_gamer'] = ((data['gm_use_dur'] > data['total_video_dur'].quantile(0.75)) & 
                            (data['gm_use_dur'] > 0)).astype(int) if is_train else 0
        
        # Streaming intensity
        data['streaming_intensity'] = (data['wtch_liv_use_dur'] + data['anchor_use_dur']) / (data['total_video_dur'] + 1)
    
    # ========================================================================
    # 7. CONTENT CONSUMPTION AND INTEREST ANALYSIS
    # ========================================================================
    
    content_cols = ['video_cnt_m', 'read_cnt_m', 'music_cnt_m', 'caijing_cnt_m', 
                    'travel_cnt_m', 'game_cnt_m', 'edu_cnt_m']
    
    if all(col in data.columns for col in content_cols):
        # Aggregated metrics
        data['total_content_cnt'] = data[content_cols].sum(axis=1)
        data['content_diversity'] = (data[content_cols] > 0).sum(axis=1)
        
        # Specific engagement metrics
        data['avg_read_time'] = data['read_time_m'] / (data['read_cnt_m'] + 1)
        data['edu_engagement'] = data['edu_time_m'] / (data['edu_cnt_m'] + 1)
        
        # Interest profiles
        data['entertainment_ratio'] = (data['video_cnt_m'] + data['music_cnt_m'] + data['game_cnt_m']) / (data['total_content_cnt'] + 1)
        data['knowledge_ratio'] = (data['read_cnt_m'] + data['edu_cnt_m'] + data['caijing_cnt_m']) / (data['total_content_cnt'] + 1)
        data['lifestyle_ratio'] = data['travel_cnt_m'] / (data['total_content_cnt'] + 1)
        
        # User persona indicators
        data['is_entertainment_focused'] = (data['entertainment_ratio'] > 0.6).astype(int)
        data['is_knowledge_seeker'] = (data['knowledge_ratio'] > 0.4).astype(int)
        data['is_diverse_user'] = (data['content_diversity'] >= 5).astype(int)
    
    # ========================================================================
    # 8. USER LABEL FEATURES
    # ========================================================================
    
    label_cols = ['hi_flux_usr_lbl', 'sev_vid_usr_lbl', 'liv_usr_lbl', 
                  'netdisk_usr_lbl', 'vid_usr_lbl', 'read_usr_lbl', 
                  'gm_usr_lbl', 'msc_usr_lbl']
    
    if all(col in data.columns for col in label_cols):
        data['total_user_labels'] = data[label_cols].sum(axis=1)
        data['is_heavy_user'] = (data['total_user_labels'] > 3).astype(int)
        data['is_video_focused'] = ((data['vid_usr_lbl'] + data['sev_vid_usr_lbl']) > 1).astype(int)
        data['is_content_creator'] = ((data['netdisk_usr_lbl'] + data['liv_usr_lbl']) > 0).astype(int)
        data['label_diversity'] = (data[label_cols] > 0).sum(axis=1)
    
    # ========================================================================
    # 9. NETWORK AND SERVICE QUALITY FEATURES
    # ========================================================================
    
    data['is_premium_user'] = ((data['is_fam_vnet_user'] == 1) | 
                               (data['is_ent_vnet_user'] == 1)).astype(int)
    data['has_unlimited'] = data['if_nulim_prod']
    data['service_stability'] = 1 - data['is_bd_status_abnormal']
    
    # Network preference
    data['is_4g_dominant'] = (data['flux_4g_use'] > 0.8 * data['cm_flux_use']).astype(int)
    
    # ========================================================================
    # 10. LOYALTY AND TENURE ANALYSIS
    # ========================================================================
    
    data['tenure_years'] = data['innet_dura'] / 365
    data['tenure_months'] = data['innet_dura'] / 30
    data['is_long_term'] = (data['innet_dura'] > 1095).astype(int)  # > 3 years
    data['is_new_customer'] = (data['innet_dura'] < 180).astype(int)  # < 6 months
    
    # Contract value and commitment
    data['contract_value'] = data['term_cont_mon'] * data['term_cont_dfee']
    data['has_contract'] = (data['term_cont_mon'] > 0).astype(int)
    data['contract_remaining_ratio'] = data['term_cont_mon'] / (data['tenure_months'] + 1)
    data['monthly_commitment'] = data['term_cont_dfee']
    data['contract_arpu_ratio'] = data['term_cont_dfee'] / (data['arpu'] + 1)
    
    # ========================================================================
    # 11. DEMOGRAPHIC AND AGE-BASED FEATURES
    # ========================================================================
    
    data['age_group'] = pd.cut(data['age'], bins=[0, 25, 35, 45, 55, 100], 
                                labels=['young', 'young_adult', 'middle', 'mature', 'senior'])
    data['is_young'] = (data['age'] < 30).astype(int)
    data['is_senior'] = (data['age'] > 55).astype(int)
    data['is_working_age'] = ((data['age'] >= 25) & (data['age'] <= 55)).astype(int)
    
    # Age-based expected behavior
    data['age_arpu_ratio'] = data['arpu'] / (data['age'] + 1)
    data['age_usage_ratio'] = data['cm_flux_use'] / (data['age'] + 1)
    
    # ========================================================================
    # 12. OUT-OF-NETWORK BEHAVIOR
    # ========================================================================
    
    data['total_out_usage'] = data['out_gprs'] + data['out_call']
    data['out_usage_ratio'] = data['total_out_usage'] / (data['cm_flux_use'] + data['cm_tot_bill_dura'] + 1)
    data['out_gprs_ratio'] = data['out_gprs'] / (data['cm_flux_use'] + 1)
    data['out_call_ratio'] = data['out_call'] / (data['cm_tot_bill_dura'] + 1)
    data['is_roamer'] = (data['total_out_usage'] > 0).astype(int)
    data['roaming_intensity'] = np.log1p(data['total_out_usage'])
    
    # ========================================================================
    # 13. ENGAGEMENT AND ACTIVITY SCORES
    # ========================================================================
    
    # Multi-dimensional engagement score
    data['engagement_score'] = (
        data['login_times_m'] * 0.2 + 
        data['click_times_m'] * 0.2 + 
        data['watch_times_m'] * 0.3 + 
        data['open_day_m'] * 0.3
    )
    
    # Activity consistency
    data['gprs_day_consistency'] = data['gprs_days'] / 30
    data['tv_day_consistency'] = data['open_day_m'] / 30
    
    # Overall digital footprint
    data['digital_footprint'] = (
        data['cm_flux_use'] / 1000 + 
        data['cm_tot_bill_dura'] / 100 + 
        data['total_content_cnt'] / 10
    )
    
    # ========================================================================
    # 14. REVENUE AND VALUE METRICS
    # ========================================================================
    
    data['revenue_per_gb'] = data['arpu'] / ((data['cm_flux_use'] / 1024) + 1)
    data['revenue_per_minute'] = data['arpu'] / (data['l3m_avg_mou'] + 1)
    data['lifetime_value'] = data['arpu'] * data['tenure_months']
    data['value_efficiency'] = data['cm_flux_use'] / (data['arpu'] + 1)
    
    # Customer profitability indicators
    data['high_value_customer'] = ((data['arpu'] > data['arpu'].quantile(0.75)) & 
                                    (data['tenure_years'] > 2)).astype(int) if is_train else 0
    data['low_value_customer'] = ((data['arpu'] < data['arpu'].quantile(0.25)) & 
                                   (data['tenure_years'] < 1)).astype(int) if is_train else 0
    
    # ========================================================================
    # 15. POLYNOMIAL AND INTERACTION FEATURES
    # ========================================================================
    
    # Key polynomial features
    data['arpu_squared'] = data['arpu'] ** 2
    data['flux_squared'] = np.log1p(data['cm_flux_use']) ** 2
    data['tenure_squared'] = data['tenure_years'] ** 2
    
    # Critical interactions
    data['arpu_tenure_interaction'] = data['arpu'] * data['tenure_years']
    data['flux_arpu_interaction'] = data['cm_flux_use'] * data['arpu'] / 1000
    data['age_tenure_interaction'] = data['age'] * data['tenure_years']
    data['contract_tenure_interaction'] = data['contract_value'] * data['tenure_years']
    
    # ========================================================================
    # 16. STATISTICAL AGGREGATIONS
    # ========================================================================
    
    # Create grouped statistics for similar features
    flux_features = ['wday_day_flux', 'wday_night_flux', 'nwday_day_flux', 'nwday_night_flux']
    if all(col in data.columns for col in flux_features):
        data['flux_mean'] = data[flux_features].mean(axis=1)
        data['flux_std'] = data[flux_features].std(axis=1)
        data['flux_max'] = data[flux_features].max(axis=1)
        data['flux_min'] = data[flux_features].min(axis=1)
        data['flux_range'] = data['flux_max'] - data['flux_min']
        data['flux_cv'] = data['flux_std'] / (data['flux_mean'] + 1)  # Coefficient of variation
    
    # ========================================================================
    # 17. CHURN RISK INDICATORS
    # ========================================================================
    
    # Declining usage indicators
    data['usage_decline_risk'] = (
        (data['flux_usage_ratio'] < 0.5) & 
        (data['gprs_day_consistency'] < 0.5)
    ).astype(int)
    
    # Contract expiry risk
    data['contract_expiry_risk'] = (
        (data['term_cont_mon'] > 0) & 
        (data['term_cont_mon'] < 3)
    ).astype(int)
    
    # Low engagement risk
    data['low_engagement_risk'] = (
        (data['engagement_score'] < data['engagement_score'].quantile(0.25)) if is_train else 0
    ).astype(int) if is_train else 0
    
    # High cost sensitivity
    data['cost_sensitive'] = (
        (data['over_plan_ratio'] > 0.3) & 
        (data['arpu'] < data['arpu'].median() if is_train else 50)
    ).astype(int)
    
    # ========================================================================
    # 18. HANDLE CATEGORICAL FEATURES
    # ========================================================================
    
    if 'age_group' in data.columns:
        le = LabelEncoder()
        data['age_group_encoded'] = le.fit_transform(data['age_group'].astype(str))
    
    # ========================================================================
    # 19. NORMALIZATION FEATURES (Log transforms for skewed distributions)
    # ========================================================================
    
    skewed_features = ['arpu', 'cm_flux_use', 'cm_tot_bill_dura', 'innet_dura']
    for feat in skewed_features:
        if feat in data.columns:
            data[f'{feat}_log'] = np.log1p(data[feat])
            data[f'{feat}_sqrt'] = np.sqrt(data[feat])
    
    def enhanced_arpu_features(data):
        # ARPU trend analysis
        data['arpu_momentum'] = data['arpu'] - data['l3m_avg_bill_dura']
        data['arpu_volatility'] = data['arpu'].rolling(3).std() if 'time' in data else 0
        data['arpu_percentile_rank'] = data['arpu'].rank(pct=True)
        
        # ARPU efficiency metrics
        data['arpu_per_service'] = data['arpu'] / (data['total_user_labels'] + 1)
        data['arpu_per_gb_efficiency'] = data['arpu'] / ((data['cm_flux_use']/1024) + 1)
        
        # Competitive ARPU positioning
        data['arpu_vs_market'] = data['arpu'] / data['arpu'].median()
        
        return data
    def advanced_temporal_features(data):
        # Seasonality patterns
        data['usage_consistency_score'] = 1 - data['flux_cv']  # Lower CV = more consistent
        data['weekend_premium_behavior'] = data['nwday_flux_total'] - data['wday_flux_total']
        
        # Time-based engagement clusters
        data['peak_usage_time'] = np.where(
            # data['total_day_flux'] > data['total_night_flux'], 'day_user',
            data['total_day_flux'] > data['total_night_flux'], -1,
            # np.where(data['total_night_flux'] > 0, 'night_user', 'minimal_user')
            np.where(data['total_night_flux'] > 0, 0, 1)
        )
        
        # Trend detection
        flux_cols = ['wday_day_flux', 'wday_night_flux', 'nwday_day_flux', 'nwday_night_flux']
        data['usage_trend'] = data[flux_cols].apply(
            lambda x: np.polyfit(range(len(x)), x, 1)[0], axis=1
        )
        
        return data
    
    def discriminative_interactions(data):
        # Multi-dimensional interactions
        data['value_engagement_score'] = (
            data['arpu'] * data['engagement_score'] * data['tenure_years']
        ) / 1000
        
        # Service portfolio analysis
        data['service_breadth'] = (
            data['has_broadband'] + data['is_premium_user'] + 
            data['has_contract'] + data['has_unlimited']
        )
        
        # Risk-weighted metrics
        data['churn_risk_composite'] = (
            data['usage_decline_risk'] * 0.3 +
            data['contract_expiry_risk'] * 0.3 +
            data['low_engagement_risk'] * 0.2 +
            data['cost_sensitive'] * 0.2
        )
        
        # Customer lifecycle stage
        data['lifecycle_stage'] = np.select([
            (data['tenure_years'] < 0.5) & (data['arpu'] < data['arpu'].median()),
            (data['tenure_years'] < 0.5) & (data['arpu'] >= data['arpu'].median()),
            (data['tenure_years'] >= 2) & (data['arpu'] >= data['arpu'].quantile(0.75)),
            (data['tenure_years'] >= 2) & (data['arpu'] < data['arpu'].quantile(0.25))
        # ], ['new_low_value', 'new_high_value', 'loyal_premium', 'loyal_budget'], 'mature')
        ], [0, 1, 2, 3], 4)
        
        return data
    
    def intelligent_feature_selection(data, target=None, top_k=50):
        # Correlation-based selection
        corr_scores = data.corrwith(target).abs().sort_values(ascending=False)
        top_corr_features = corr_scores.head(top_k).index.tolist()
        
        # Variance-based filtering (remove low variance)
        variance_threshold = VarianceThreshold(threshold=0.01)
        high_var_features = data.columns[variance_threshold.fit(data).get_support()]
        
        # Mutual information selection
        mi_scores = mutual_info_classif(data, target, random_state=42)
        mi_features = data.columns[np.argsort(mi_scores)[-top_k:]].tolist()
        
        # Combine selections
        selected_features = list(set(top_corr_features + mi_features) & set(high_var_features))
        
        return selected_features
    
    def category_specific_features(data):
        # Enhanced usage patterns
        data['usage_efficiency_index'] = (
            data['cm_flux_use'] / data['arpu'] * 
            data['gprs_day_consistency']
        )
        
        # # Behavioral change detection
        # data['behavior_change_indicator'] = abs(
        #     data['recent_usage'] - data['historical_usage']
        # ) / (data['historical_usage'] + 1)
        
        # Content preference stability
        content_features = ['video_cnt_m', 'read_cnt_m', 'music_cnt_m', 'game_cnt_m']
        data['content_preference_stability'] = 1 - data[content_features].std(axis=1) / (data[content_features].mean(axis=1) + 1)
        
        return data
    
    def statistical_feature_engineering(data):
        # Robust scaling for extreme values
        from sklearn.preprocessing import RobustScaler
        
        # # Z-score based outlier features
        # for col in ['arpu', 'cm_flux_use', 'cm_tot_bill_dura']:
        #     z_scores = np.abs(stats.zscore(data[col]))
        #     data[f'{col}_is_outlier'] = (z_scores > 3).astype(int)
        #     data[f'{col}_outlier_magnitude'] = z_scores
        
        # Distribution-based features
        data['arpu_quartile'] = pd.qcut(data['arpu'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
        data['usage_decile'] = pd.qcut(data['cm_flux_use'], q=10, labels=False, duplicates='drop')
        
        # Relative positioning features
        data['arpu_rank_percentile'] = data['arpu'].rank(pct=True)
        data['usage_rank_percentile'] = data['cm_flux_use'].rank(pct=True)
        
        return data
    
    def business_logic_features(data):
        # Customer value segmentation
        data['customer_segment'] = np.select([
            (data['arpu'] >= data['arpu'].quantile(0.8)) & (data['tenure_years'] >= 2),
            (data['arpu'] >= data['arpu'].quantile(0.6)) & (data['tenure_years'] >= 1),
            (data['arpu'] <= data['arpu'].quantile(0.3)) & (data['tenure_years'] <= 0.5),
        # ], ['premium_loyal', 'value_stable', 'price_sensitive'], 'standard')
        ], [2, 1, 0], 3)
        
        # Churn probability indicators
        data['early_warning_score'] = (
            (data['usage_decline_risk'] * 0.25) +
            (data['low_engagement_risk'] * 0.25) +
            (data['contract_expiry_risk'] * 0.25) +
            (data['cost_sensitive'] * 0.25)
        )
        
        # Competitive vulnerability
        data['competitive_vulnerability'] = (
            (data['arpu'] > data['arpu'].quantile(0.75)) & 
            (data['contract_remaining_ratio'] < 0.2) &
            (data['usage_decline_risk'] == 1)
        ).astype(int)
        
        return data
    
    if enhancement == True:
        data = enhanced_arpu_features(data)
        data = advanced_temporal_features(data)
        data = discriminative_interactions(data)
        if is_train and 'label' in data.columns:
            target = data['label'].fillna(0)
            candidate_features = data.drop(columns=['label', 'id', 'age_group', 'peak_usage_time', 'lifecycle_stage']).fillna(0)
            selected_feats = intelligent_feature_selection(candidate_features, target)
            data = data[selected_feats + ['label', 'id', 'age_group', 'peak_usage_time', 'lifecycle_stage']]
        else:
            data = category_specific_features(data)
            data = statistical_feature_engineering(data)
            data = business_logic_features(data)

    return data

def get_feature_columns(df):
    """Get the list of feature columns to use for modeling"""
    exclude_cols = ['Unnamed: 0', 'id', 'label', 'age_group']
    feature_cols = [col for col in df.columns if col not in exclude_cols]
    return feature_cols

def reduce_features_by_importance(X, y, model, threshold=0.001):
    """
    Reduce features based on feature importance
    """
    print(f"\nPerforming feature selection (threshold={threshold})...")
    selector = SelectFromModel(model, threshold=threshold, prefit=False)
    selector.fit(X, y)
    selected_features = X.columns[selector.get_support()].tolist()
    print(f"Selected {len(selected_features)} features out of {len(X.columns)}")
    return selected_features

# Define custom scoring function
def custom_score(y_true, y_pred, y_proba=None):
    """Calculate custom score: 0.7 * accuracy + 0.3 * f1"""
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    return 0.7 * acc + 0.3 * f1

custom_scorer = make_scorer(custom_score, needs_proba=False)


from sklearn.metrics import precision_recall_curve

def find_optimal_threshold(y_true, y_proba, metric='f1'):
    """Find optimal threshold for F1 score"""
    precision, recall, thresholds = precision_recall_curve(y_true, y_proba)
    f1_scores = 2 * (precision * recall) / (precision + recall)
    
    # Handle NaN values
    f1_scores = np.nan_to_num(f1_scores)
    
    optimal_idx = np.argmax(f1_scores)
    optimal_threshold = thresholds[optimal_idx]
    optimal_f1 = f1_scores[optimal_idx]
    
    return optimal_threshold, optimal_f1

# ============================================================================
# LOAD AND PREPARE TRAINING DATA
# ============================================================================
print("=" * 80)
print("LOADING TRAINING DATA")
print("=" * 80)

df_train = pd.read_csv('./train.csv')
print(f"Training data shape: {df_train.shape}")
print(f"\nClass distribution:")
print(df_train['label'].value_counts())
print(f"Class ratio: {df_train['label'].value_counts()[0] / df_train['label'].value_counts()[1]:.2f}:1")

# Check for missing values
print(f"\nMissing values summary:")
missing_summary = df_train.isnull().sum()
missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)
if len(missing_summary) > 0:
    print(missing_summary.head(10))
else:
    print("No missing values found")

# ============================================================================
# ADVANCED FEATURE ENGINEERING
# ============================================================================
print("\n" + "=" * 80)
print("CREATING ADVANCED FEATURES")
print("=" * 80)

df_train_processed = create_advanced_features(df_train, is_train=True)

# Get feature columns
feature_cols = get_feature_columns(df_train_processed)
print(f"Total features after engineering: {len(feature_cols)}")

# Clean feature names
original_feature_cols = feature_cols.copy()
cleaned_feature_cols = clean_feature_names(feature_cols)
feature_name_mapping = dict(zip(cleaned_feature_cols, original_feature_cols))

# Prepare X and y
X = df_train_processed[feature_cols].copy()
X.columns = cleaned_feature_cols
y = df_train_processed['label']

# Handle missing values
print("\nHandling missing values...")
for col in X.columns:
    if X[col].dtype in ['float64', 'int64']:
        X[col].fillna(X[col].median(), inplace=True)
    else:
        X[col].fillna(X[col].mode()[0] if len(X[col].mode()) > 0 else 0, inplace=True)

# Replace infinities
X.replace([np.inf, -np.inf], 0, inplace=True)

print(f"Final feature matrix shape: {X.shape}")
print(f"\nFeature statistics:")
print(f"  Numeric features: {X.select_dtypes(include=[np.number]).shape[1]}")
print(f"  Memory usage: {X.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# ============================================================================
# FEATURE SCALING (Optional - for some models)
# ============================================================================
print("\n" + "=" * 80)
print("FEATURE SCALING")
print("=" * 80)

# Create scaled version for models that benefit from it
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns,
    index=X.index
)
print("Features scaled using StandardScaler")

# ============================================================================
# TRAIN-VALIDATION SPLIT
# ============================================================================
print("\n" + "=" * 80)
print("SPLITTING DATA FOR VALIDATION")
print("=" * 80)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Training class distribution: {y_train.value_counts().to_dict()}")
print(f"Validation class distribution: {y_val.value_counts().to_dict()}")


LOADING TRAINING DATA
Training data shape: (59904, 88)

Class distribution:
label
0    44904
1    15000
Name: count, dtype: int64
Class ratio: 2.99:1

Missing values summary:
gm_use_dur               47451
gm_dayt_use_dur          47451
gm_ngt_use_dur           47451
shrt_vid_use_dur         47451
shrt_vid_dayt_use_dur    47451
shrt_vid_ngt_use_dur     47451
long_vid_use_dur         47451
long_vid_dayt_use_dur    47451
long_vid_ngt_use_dur     47451
anchor_use_dur           47451
dtype: int64

CREATING ADVANCED FEATURES
Total features after engineering: 73

Handling missing values...
Final feature matrix shape: (59904, 73)

Feature statistics:
  Numeric features: 73
  Memory usage: 33.36 MB

FEATURE SCALING
Features scaled using StandardScaler

SPLITTING DATA FOR VALIDATION
Training set: (47923, 73)
Validation set: (11981, 73)
Training class distribution: {0: 35923, 1: 12000}
Validation class distribution: {0: 8981, 1: 3000}


In [5]:

# ============================================================================
# ADVANCED MODEL TRAINING WITH CROSS-VALIDATION
# ============================================================================
print("\n" + "=" * 80)
print("TRAINING ADVANCED MODELS WITH CROSS-VALIDATION")
print("=" * 80)

# Calculate scale_pos_weight
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Scale pos weight: {scale_pos_weight:.2f}")

# Setup cross-validation
n_folds = 5
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

# Store results
results = {}



TRAINING ADVANCED MODELS WITH CROSS-VALIDATION
Scale pos weight: 2.99


In [6]:

# ============================================================================
# MODEL 1: LIGHTGBM WITH HYPERPARAMETER TUNING
# ============================================================================
print("\n" + "-" * 80)
print("1. TRAINING LIGHTGBM")
print("-" * 80)

lgb_params = {
    'n_estimators': 1000,
    'max_depth': 10,
    'learning_rate': 0.03,
    'num_leaves': 40,
    'subsample': 0.8,
    'colsample_bytree': 0.7,
    'min_child_samples': 30,
    'reg_alpha': 0.3,
    'reg_lambda': 0.3,
    'class_weight': 'balanced',
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1,
    'force_col_wise': True,
    'min_split_gain': 0.01,
    'min_child_weight': 0.001
}
lgb_params_enhanced = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'n_estimators': 1500,  # Increased
    'max_depth': 12,       # Slightly deeper
    'learning_rate': 0.02, # Lower learning rate
    'num_leaves': 60,      # More leaves
    'subsample': 0.85,
    'subsample_freq': 1,
    'colsample_bytree': 0.8,
    'min_child_samples': 25,
    'min_child_weight': 0.001,
    'min_split_gain': 0.005,
    'reg_alpha': 0.4,
    'reg_lambda': 0.4,
    # 'class_weight': 'balanced',
    'random_state': 42,
    'feature_fraction_seed': 42,
    'bagging_seed': 42,
    'drop_seed': 42,
    'data_random_seed': 42,
    'extra_trees': True,      # Extra randomization
    'path_smooth': 0.1,       # Path smoothing
    'verbose': -1,
    'force_col_wise': True
}

lgb_model = lgb.LGBMClassifier(**lgb_params_enhanced)

# # Cross-validation
# print("Performing 5-fold cross-validation...")
# cv_scores_lgb = cross_val_score(lgb_model, X_train, y_train, cv=skf, 
#                                  scoring='roc_auc', n_jobs=-1)
# print(f"CV ROC-AUC scores: {cv_scores_lgb}")
# print(f"Mean CV ROC-AUC: {cv_scores_lgb.mean():.4f} (+/- {cv_scores_lgb.std() * 2:.4f})")

# Train on full training set with early stopping
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
)

y_pred_lgb = lgb_model.predict(X_val)
y_proba_lgb = lgb_model.predict_proba(X_val)[:, 1]


lgb_acc = accuracy_score(y_val, y_pred_lgb)
lgb_auc = roc_auc_score(y_val, y_proba_lgb)
lgb_f1 = f1_score(y_val, y_pred_lgb)
custom_score_lgb = custom_score(y_val, y_pred_lgb, y_proba_lgb)

print(f"\nValidation Metrics:")
print(f"  Accuracy: {lgb_acc:.4f}")
print(f"  ROC-AUC: {lgb_auc:.4f}")
print(f"  F1-Score: {lgb_f1:.4f}")
print(f"  Custom Score: {custom_score_lgb:.4f}")

results['LightGBM'] = {
    'model': lgb_model,
    # 'cv_scores': cv_scores_lgb,
    'val_auc': lgb_auc,
    'val_f1': lgb_f1,
    'predictions': y_pred_lgb,
    'probabilities': y_proba_lgb,
    'custom_score': custom_score_lgb
}

# ============================================================================
# MODEL 1.1: FocalLossLGBM WITH HYPERPARAMETER TUNING
# ============================================================================
print("\n" + "-" * 80)
print("1.1. TRAINING FocalLossLGBM (FIXED)")
print("-" * 80)

import numpy as np
import lightgbm as lgb

def focal_loss_lgb(y_pred, y_true, alpha=0.25, gamma=2.0):
    """
    Focal Loss for LightGBM
    """
    # Get the true labels
    y_true = y_true.get_label()
    
    # Apply sigmoid to get probabilities
    p = 1 / (1 + np.exp(-y_pred))
    
    # Calculate focal loss components
    # For positive samples (y_true = 1)
    pos_loss = -alpha * ((1 - p) ** gamma) * np.log(np.clip(p, 1e-8, 1.0))
    # For negative samples (y_true = 0)  
    neg_loss = -(1 - alpha) * (p ** gamma) * np.log(np.clip(1 - p, 1e-8, 1.0))
    
    # Combine losses
    loss = np.where(y_true == 1, pos_loss, neg_loss)
    
    # Calculate gradients
    # Positive samples
    grad_pos = alpha * (1 - p) ** (gamma - 1) * (gamma * p * np.log(np.clip(p, 1e-8, 1.0)) + p - 1)
    # Negative samples
    grad_neg = (1 - alpha) * p ** (gamma - 1) * (gamma * (1 - p) * np.log(np.clip(1 - p, 1e-8, 1.0)) + p)
    
    grad = np.where(y_true == 1, grad_pos, grad_neg)
    
    # Calculate hessians (simplified)
    hess = np.abs(grad) + 1e-6
    
    return grad, hess

# Alternative: Use class_weight parameter instead of custom objective
print("Testing with class weights first...")

# Calculate class weights
from sklearn.utils.class_weight import compute_class_weight
classes = np.unique(y_train)
class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = {classes[i]: class_weights[i] for i in range(len(classes))}

print(f"Class weights: {class_weight_dict}")

# Method 1: Using class weights (simpler and often more stable)
lgb_focal_weighted = lgb.LGBMClassifier(
    **lgb_params_enhanced,
    class_weight=class_weight_dict,  # or use class_weight_dict
    # random_state=42
)

lgb_focal_weighted.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
)

y_pred_focal_weighted = lgb_focal_weighted.predict(X_val)
y_proba_focal_weighted = lgb_focal_weighted.predict_proba(X_val)[:, 1]

focal_weighted_acc = accuracy_score(y_val, y_pred_focal_weighted)
focal_weighted_auc = roc_auc_score(y_val, y_proba_focal_weighted)
focal_weighted_f1 = f1_score(y_val, y_pred_focal_weighted)
custom_score_focal_weighted = custom_score(y_val, y_pred_focal_weighted, y_proba_focal_weighted)

print(f"\nValidation Metrics for FocalLossLGBM (Class Weighted):")
print(f"  Accuracy: {focal_weighted_acc:.4f}")
print(f"  ROC-AUC: {focal_weighted_auc:.4f}")
print(f"  F1-Score: {focal_weighted_f1:.4f}")
print(f"  Custom Score: {custom_score_focal_weighted:.4f}")

# Method 2: Using custom focal loss objective (if you want to try)
print(f"\nTesting custom focal loss objective...")

# Create dataset for custom objective
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

# Parameters for custom objective
focal_params = {
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'random_state': 42,
    'n_estimators': 1000
}

try:
    # Train with custom focal loss
    focal_model = lgb.train(
        focal_params,
        train_data,
        valid_sets=[val_data],
        fobj=focal_loss_lgb,
        feval=None,
        num_boost_round=1000,
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    y_pred_focal_custom = focal_model.predict(X_val)
    y_pred_focal_custom_binary = (y_pred_focal_custom > 0).astype(int)
    
    # Convert predictions to probabilities
    y_proba_focal_custom = 1 / (1 + np.exp(-y_pred_focal_custom))
    
    focal_custom_acc = accuracy_score(y_val, y_pred_focal_custom_binary)
    focal_custom_auc = roc_auc_score(y_val, y_proba_focal_custom)
    focal_custom_f1 = f1_score(y_val, y_pred_focal_custom_binary)
    custom_score_focal_custom = custom_score(y_val, y_pred_focal_custom_binary, y_proba_focal_custom)
    
    print(f"\nValidation Metrics for FocalLossLGBM (Custom Objective):")
    print(f"  Accuracy: {focal_custom_acc:.4f}")
    print(f"  ROC-AUC: {focal_custom_auc:.4f}")
    print(f"  F1-Score: {focal_custom_f1:.4f}")
    print(f"  Custom Score: {custom_score_focal_custom:.4f}")
    
except Exception as e:
    print(f"Custom focal loss failed: {e}")
    print("Using class weighted version as focal loss alternative...")
    focal_custom_acc = focal_weighted_acc
    focal_custom_auc = focal_weighted_auc
    focal_custom_f1 = focal_weighted_f1
    custom_score_focal_custom = custom_score_focal_weighted
    focal_model = lgb_focal_weighted

# Store the better performing model
if focal_weighted_auc >= focal_custom_auc:
    print(f"\nUsing class-weighted version (better performance)")
    best_focal_model = lgb_focal_weighted
    best_focal_pred = y_pred_focal_weighted
    best_focal_proba = y_proba_focal_weighted
    best_focal_metrics = {
        'accuracy': focal_weighted_acc,
        'auc': focal_weighted_auc,
        'f1': focal_weighted_f1,
        'custom': custom_score_focal_weighted
    }
else:
    print(f"\nUsing custom objective version (better performance)")
    best_focal_model = focal_model
    best_focal_pred = y_pred_focal_custom_binary
    best_focal_proba = y_proba_focal_custom
    best_focal_metrics = {
        'accuracy': focal_custom_acc,
        'auc': focal_custom_auc,
        'f1': focal_custom_f1,
        'custom': custom_score_focal_custom
    }

results['FocalLossLGBM'] = {
    'model': best_focal_model,
    'val_auc': best_focal_metrics['auc'],
    'val_f1': best_focal_metrics['f1'],
    'predictions': best_focal_pred,
    'probabilities': best_focal_proba,
    'custom_score': best_focal_metrics['custom']
}

print(f"\nFinal FocalLossLGBM Results:")
print(f"  Accuracy: {best_focal_metrics['accuracy']:.4f}")
print(f"  ROC-AUC: {best_focal_metrics['auc']:.4f}")
print(f"  F1-Score: {best_focal_metrics['f1']:.4f}")
print(f"  Custom Score: {best_focal_metrics['custom']:.4f}")



--------------------------------------------------------------------------------
1. TRAINING LIGHTGBM
--------------------------------------------------------------------------------

Validation Metrics:
  Accuracy: 0.7959
  ROC-AUC: 0.8324
  F1-Score: 0.5330
  Custom Score: 0.7170

--------------------------------------------------------------------------------
1.1. TRAINING FocalLossLGBM (FIXED)
--------------------------------------------------------------------------------
Testing with class weights first...
Class weights: {np.int64(0): np.float64(0.6670239122567714), np.int64(1): np.float64(1.9967916666666667)}

Validation Metrics for FocalLossLGBM (Class Weighted):
  Accuracy: 0.7554
  ROC-AUC: 0.8316
  F1-Score: 0.6113
  Custom Score: 0.7122

Testing custom focal loss objective...
Custom focal loss failed: train() got an unexpected keyword argument 'fobj'
Using class weighted version as focal loss alternative...

Using class-weighted version (better performance)

Final FocalLos

In [7]:
# ============================================================================
# MODEL 2: XGBOOST WITH ADVANCED PARAMETERS
# ============================================================================
print("\n" + "-" * 80)
print("2. TRAINING XGBOOST")
print("-" * 80)

xgb_params = {
    'n_estimators': 1000,
    'max_depth': 7,
    'learning_rate': 0.03,
    'subsample': 0.8,
    'colsample_bytree': 0.7,
    'colsample_bylevel': 0.7,
    'colsample_bynode': 0.7,
    'min_child_weight': 5,
    'gamma': 0.1,
    'reg_alpha': 0.3,
    'reg_lambda': 0.3,
    'scale_pos_weight': scale_pos_weight,
    'random_state': 42,
    'n_jobs': -1,
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'max_bin': 256
}
xgb_params_enhanced = {
    'n_estimators': 1500,
    'max_depth': 8,
    'learning_rate': 0.02,
    'subsample': 0.85,
    'colsample_bytree': 0.8,
    'colsample_bylevel': 0.8,
    'colsample_bynode': 0.8,
    'min_child_weight': 3,
    'gamma': 0.05,
    'reg_alpha': 0.4,
    'reg_lambda': 0.4,
    'scale_pos_weight': scale_pos_weight,
    'random_state': 42,
    'tree_method': 'hist',
    'grow_policy': 'depthwise',
    'max_bin': 512,          # Increased bins
    # 'sampling_method': 'gradient_based',
    'eval_metric': 'auc'
}

xgb_model = xgb.XGBClassifier(**xgb_params_enhanced)

# # Cross-validation
# print("Performing 5-fold cross-validation...")
# cv_scores_xgb = cross_val_score(xgb_model, X_train, y_train, cv=skf, 
#                                  scoring='roc_auc', n_jobs=-1)
# print(f"CV ROC-AUC scores: {cv_scores_xgb}")
# print(f"Mean CV ROC-AUC: {cv_scores_xgb.mean():.4f} (+/- {cv_scores_xgb.std() * 2:.4f})")

# Train on full training set with early stopping
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    # early_stopping_rounds=50,
    verbose=False
)

y_pred_xgb = xgb_model.predict(X_val)
y_proba_xgb = xgb_model.predict_proba(X_val)[:, 1]

xgb_acc = accuracy_score(y_val, y_pred_xgb)
xgb_auc = roc_auc_score(y_val, y_proba_xgb)
xgb_f1 = f1_score(y_val, y_pred_xgb)
xgb_custom_score = custom_score(y_val, y_pred_xgb, y_proba_xgb)

print(f"\nValidation Metrics:")
print(f"  Accuracy: {xgb_acc:.4f}")
print(f"  ROC-AUC: {xgb_auc:.4f}")
print(f"  F1-Score: {xgb_f1:.4f}")
print(f"  Custom Score: {xgb_custom_score:.4f}")

results['XGBoost'] = {
    'model': xgb_model,
    # 'cv_scores': cv_scores_xgb,
    'val_auc': xgb_auc,
    'val_f1': xgb_f1,
    'predictions': y_pred_xgb,
    'probabilities': y_proba_xgb,
    'custom_score': xgb_custom_score
    
}


--------------------------------------------------------------------------------
2. TRAINING XGBOOST
--------------------------------------------------------------------------------

Validation Metrics:
  Accuracy: 0.7872
  ROC-AUC: 0.8376
  F1-Score: 0.6201
  Custom Score: 0.7370


In [8]:
# ============================================================================
# MODEL 3: CATBOOST
# ============================================================================
print("\n" + "-" * 80)
print("3. TRAINING CATBOOST")
print("-" * 80)

catboost_params = {
    'iterations': 1000,
    'depth': 8,
    'learning_rate': 0.03,
    'l2_leaf_reg': 3,
    'subsample': 0.8,
    'colsample_bylevel': 0.8,
    'min_data_in_leaf': 20,
    'auto_class_weights': 'Balanced',
    'random_state': 42,
    'thread_count': -1,
    'verbose': False,
    'early_stopping_rounds': 50,
    'eval_metric': 'AUC',
    'border_count': 128
}

catboost_model = CatBoostClassifier(**catboost_params)

# # Cross-validation
# print("Performing 5-fold cross-validation...")
# cv_scores_cat = cross_val_score(catboost_model, X_train, y_train, cv=skf, 
#                                 scoring='roc_auc', n_jobs=-1)
# print(f"CV ROC-AUC scores: {cv_scores_cat}")
# print(f"Mean CV ROC-AUC: {cv_scores_cat.mean():.4f} (+/- {cv_scores_cat.std() * 2:.4f})")

# Train on full training set
catboost_model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    verbose=False
)

y_pred_cat = catboost_model.predict(X_val)
y_proba_cat = catboost_model.predict_proba(X_val)[:, 1]

cat_acc = accuracy_score(y_val, y_pred_cat)
cat_auc = roc_auc_score(y_val, y_proba_cat)
cat_f1 = f1_score(y_val, y_pred_cat)
cat_custom_score = custom_score(y_val, y_pred_cat, y_proba_cat)

print(f"\nValidation Metrics:")
print(f"  Accuracy: {cat_acc:.4f}")
print(f"  ROC-AUC: {cat_auc:.4f}")
print(f"  F1-Score: {cat_f1:.4f}")
print(f"  Custom Score: {cat_custom_score:.4f}")

results['CatBoost'] = {
    'model': catboost_model,
    # 'cv_scores': cv_scores_cat,
    'val_auc': cat_auc,
    'val_f1': cat_f1,
    'predictions': y_pred_cat,
    'probabilities': y_proba_cat,
    'custom_score': cat_custom_score
}



--------------------------------------------------------------------------------
3. TRAINING CATBOOST
--------------------------------------------------------------------------------

Validation Metrics:
  Accuracy: 0.7614
  ROC-AUC: 0.8369
  F1-Score: 0.6197
  Custom Score: 0.7189


In [9]:
# ============================================================================
# MODEL 4: RANDOM FOREST WITH OPTIMIZED PARAMETERS
# ============================================================================
print("\n" + "-" * 80)
print("4. TRAINING RANDOM FOREST")
print("-" * 80)

rf_params = {
    'n_estimators': 500,
    'max_depth': 15,
    'min_samples_split': 10,
    'min_samples_leaf': 5,
    'max_features': 'sqrt',
    'bootstrap': True,
    'class_weight': y_train.value_counts().to_dict(),
    'random_state': 42,
    'n_jobs': -1,
    'criterion': 'gini',
    'max_samples': 0.8
}

rf_model = RandomForestClassifier(**rf_params)

# # Cross-validation
# print("Performing 5-fold cross-validation...")
# cv_scores_rf = cross_val_score(rf_model, X_train, y_train, cv=skf, 
#                                scoring='roc_auc', n_jobs=-1)
# print(f"CV ROC-AUC scores: {cv_scores_rf}")
# print(f"Mean CV ROC-AUC: {cv_scores_rf.mean():.4f} (+/- {cv_scores_rf.std() * 2:.4f})")

# Train on full training set
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_val)
y_proba_rf = rf_model.predict_proba(X_val)[:, 1]

rf_acc = accuracy_score(y_val, y_pred_rf)
rf_auc = roc_auc_score(y_val, y_proba_rf)
rf_f1 = f1_score(y_val, y_pred_rf)
rf_custom_score = custom_score(y_val, y_pred_rf, y_proba_rf)

print(f"\nValidation Metrics:")
print(f"  Accuracy: {rf_acc:.4f}")
print(f"  ROC-AUC: {rf_auc:.4f}")
print(f"  F1-Score: {rf_f1:.4f}")
print(f"  Custom Score: {rf_custom_score:.4f}")

results['RandomForest'] = {
    'model': rf_model,
    # 'cv_scores': cv_scores_rf,
    'val_auc': rf_auc,
    'val_f1': rf_f1,
    'predictions': y_pred_rf,
    'probabilities': y_proba_rf,
    'custom_score': rf_custom_score
}


--------------------------------------------------------------------------------
4. TRAINING RANDOM FOREST
--------------------------------------------------------------------------------

Validation Metrics:
  Accuracy: 0.7647
  ROC-AUC: 0.8309
  F1-Score: 0.1745
  Custom Score: 0.5877


In [10]:

# ============================================================================
# ENSEMBLE METHODS
# ============================================================================
print("\n" + "=" * 80)
print("CREATING ENSEMBLE MODELS")
print("=" * 80)

# Weighted Average Ensemble
print("\n" + "-" * 80)
print("1. WEIGHTED AVERAGE ENSEMBLE")
print("-" * 80)

# Use validation AUC scores as weights
weights = np.array([
    results['LightGBM']['custom_score'],
    results['FocalLossLGBM']['custom_score'],
    results['XGBoost']['custom_score'],
    results['CatBoost']['custom_score'],
    results['RandomForest']['custom_score']
])
weights = weights / weights.sum()  # Normalize weights

print(f"Ensemble weights: LGB={weights[0]:.3f}, XGB={weights[1]:.3f}, "
      f"CAT={weights[2]:.3f}, RF={weights[3]:.3f}")

# Create weighted ensemble predictions
ensemble_proba = (weights[0] * results['LightGBM']['probabilities'] +
                 weights[1] * results['XGBoost']['probabilities'] +
                 weights[2] * results['CatBoost']['probabilities'] +
                 weights[3] * results['RandomForest']['probabilities'])

ensemble_pred = (ensemble_proba > 0.5).astype(int)

ensemble_acc = accuracy_score(y_val, ensemble_pred)
ensemble_auc = roc_auc_score(y_val, ensemble_proba)
ensemble_f1 = f1_score(y_val, ensemble_pred)
ensemble_custom_score = custom_score(y_val, ensemble_pred, ensemble_proba)

print(f"\nWeighted Ensemble Metrics:")
print(f"  Accuracy: {ensemble_acc:.4f}")
print(f"  ROC-AUC: {ensemble_auc:.4f}")
print(f"  F1-Score: {ensemble_f1:.4f}")
print(f"  Custom Score: {ensemble_custom_score:.4f}")

results['WeightedEnsemble'] = {
    'val_auc': ensemble_auc,
    'val_f1': ensemble_f1,
    'predictions': ensemble_pred,
    'probabilities': ensemble_proba,
    'weights': weights,
    'custom_score': ensemble_custom_score
}

class F1OptimizedEnsemble:
    def __init__(self):
        self.models = []
        self.weights = []
        self.thresholds = []
    
    def add_model(self, model, X_val, y_val):
        """Add model with optimized threshold"""
        y_proba = model.predict_proba(X_val)
        y_proba = y_proba[:, 1] if y_proba.ndim > 1 else y_proba
        threshold, f1 = find_optimal_threshold(y_val, y_proba)
        
        self.models.append(model)
        self.thresholds.append(threshold)
        self.weights.append(f1)  # Weight by F1 performance
    
    def predict(self, X):
        predictions = []
        for model, threshold in zip(self.models, self.thresholds):
            y_proba = model.predict_proba(X)[:, 1]
            y_pred = (y_proba >= threshold).astype(int)
            predictions.append(y_pred)
        
        # Weighted voting
        weighted_preds = np.average(predictions, axis=0, weights=self.weights)
        return (weighted_preds >= 0.5).astype(int)

# Usage
f1_ensemble = F1OptimizedEnsemble()
for model_name, model_info in results.items():
    f1_ensemble.add_model(model_info['model'], X_val, y_val) if 'model' in model_info else None

y_pred_f1_ensemble = f1_ensemble.predict(X_val)

f1_ensemble_acc = accuracy_score(y_val, y_pred_f1_ensemble)
f1_ensemble_auc = roc_auc_score(y_val, y_pred_f1_ensemble)
f1_ensemble_f1 = f1_score(y_val, y_pred_f1_ensemble)
f1_ensemble_custom_score = custom_score(y_val, y_pred_f1_ensemble)

print(f"\nF1-Optimized Ensemble Metrics:")
print(f"  Accuracy: {f1_ensemble_acc:.4f}")
print(f"  ROC-AUC: {f1_ensemble_auc:.4f}")
print(f"  F1-Score: {f1_ensemble_f1:.4f}")
print(f"  Custom Score: {f1_ensemble_custom_score:.4f}")
results['F1OptimizedEnsemble'] = {
    'val_auc': f1_ensemble_auc,
    'val_f1': f1_ensemble_f1,
    'predictions': y_pred_f1_ensemble,
    'custom_score': f1_ensemble_custom_score
}



CREATING ENSEMBLE MODELS

--------------------------------------------------------------------------------
1. WEIGHTED AVERAGE ENSEMBLE
--------------------------------------------------------------------------------
Ensemble weights: LGB=0.206, XGB=0.205, CAT=0.212, RF=0.207

Weighted Ensemble Metrics:
  Accuracy: 0.7973
  ROC-AUC: 0.8404
  F1-Score: 0.4832
  Custom Score: 0.7031

F1-Optimized Ensemble Metrics:
  Accuracy: 0.7792
  ROC-AUC: 0.7666
  F1-Score: 0.6271
  Custom Score: 0.7336


In [11]:

# Voting Classifier
print("\n" + "-" * 80)
print("2. VOTING CLASSIFIER")
print("-" * 80)
weights_voting = weights = np.array([
    # results['LightGBM']['custom_score'],
    results['XGBoost']['custom_score'],
    results['CatBoost']['custom_score'],
    results['RandomForest']['custom_score']
])
voting_estimators = [
    # ('lgb', lgb_model),
    ('xgb', xgb_model),
    ('cat', catboost_model),
    ('rf', rf_model),
]

voting_clf = VotingClassifier(
    estimators=voting_estimators,
    voting='soft',
    weights=weights,
    n_jobs=-1
)

# # Cross-validation for voting classifier
# cv_scores_voting = cross_val_score(voting_clf, X_train, y_train, cv=skf, 
#                                   scoring='roc_auc', n_jobs=-1)
# print(f"CV ROC-AUC scores: {cv_scores_voting}")
# print(f"Mean CV ROC-AUC: {cv_scores_voting.mean():.4f} (+/- {cv_scores_voting.std() * 2:.4f})")

# Train voting classifier
voting_clf.fit(X_train, y_train)

y_pred_voting = voting_clf.predict(X_val)
y_proba_voting = voting_clf.predict_proba(X_val)[:, 1]

voting_acc = accuracy_score(y_val, y_pred_voting)
voting_auc = roc_auc_score(y_val, y_proba_voting)
voting_f1 = f1_score(y_val, y_pred_voting)
voting_custom_score = custom_score(y_val, y_pred_voting, y_proba_voting)

print(f"\nVoting Classifier Metrics:")
print(f"  Accuracy: {voting_acc:.4f}")
print(f"  ROC-AUC: {voting_auc:.4f}")
print(f"  F1-Score: {voting_f1:.4f}")
print(f"  Custom Score: {voting_custom_score:.4f}")

results['VotingClassifier'] = {
    'model': voting_clf,
    # 'cv_scores': cv_scores_voting,
    'val_auc': voting_auc,
    'val_f1': voting_f1,
    'predictions': y_pred_voting,
    'probabilities': y_proba_voting,
    'custom_score': voting_custom_score
}



--------------------------------------------------------------------------------
2. VOTING CLASSIFIER
--------------------------------------------------------------------------------

Voting Classifier Metrics:
  Accuracy: 0.7992
  ROC-AUC: 0.8411
  F1-Score: 0.6097
  Custom Score: 0.7423


In [12]:


# STACKING CLASSIFIER
print("\n" + "-" * 80)
print("3. STACKING CLASSIFIER")
print("-" * 80)

from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

stacking_clf = StackingClassifier(
    estimators=voting_estimators,
    final_estimator=LogisticRegression(max_iter=1000, class_weight='balanced'),
    cv=5,
    n_jobs=-1,
    passthrough=False
)

# # Cross-validation for stacking classifier
# cv_scores_stacking = cross_val_score(stacking_clf, X_train, y_train, cv=skf, 
#                                   scoring='roc_auc', n_jobs=-1)
# print(f"CV ROC-AUC scores: {cv_scores_stacking}")
# print(f"Mean CV ROC-AUC: {cv_scores_stacking.mean():.4f} (+/- {cv_scores_stacking.std() * 2:.4f})")

# Train stacking classifier
stacking_clf.fit(X_train, y_train)

y_pred_stacking = stacking_clf.predict(X_val)
y_proba_stacking = stacking_clf.predict_proba(X_val)[:, 1]

stacking_acc = accuracy_score(y_val, y_pred_stacking)
stacking_auc = roc_auc_score(y_val, y_proba_stacking)
stacking_f1 = f1_score(y_val, y_pred_stacking)
stacking_custom_score = custom_score(y_val, y_pred_stacking, y_proba_stacking)

print(f"\nstacking Classifier Metrics:")
print(f"  Accuracy: {stacking_acc:.4f}")
print(f"  ROC-AUC: {stacking_auc:.4f}")
print(f"  F1-Score: {stacking_f1:.4f}")
print(f"  Custom Score: {stacking_custom_score:.4f}")

results['stackingClassifier'] = {
    'model': stacking_clf,
    # 'cv_scores': cv_scores_stacking,
    'val_auc': stacking_auc,
    'val_f1': stacking_f1,
    'predictions': y_pred_stacking,
    'probabilities': y_proba_stacking,
    'custom_score': stacking_custom_score
}




--------------------------------------------------------------------------------
3. STACKING CLASSIFIER
--------------------------------------------------------------------------------

stacking Classifier Metrics:
  Accuracy: 0.7680
  ROC-AUC: 0.8408
  F1-Score: 0.6242
  Custom Score: 0.7248


In [16]:
# Multi-level STACKING CLASSIFIER
print("\n" + "-" * 80)
print("4. Multi-level STACKING CLASSIFIER")
print("-" * 80)

from sklearn import clone
from sklearn.model_selection import KFold
from sklearn.svm import SVC

class EnhancedMultiLevelStacking:
    def __init__(self, focus_metric='f1'):
        self.focus_metric = focus_metric
        
        # Level 1: More diverse base models with different strengths
        self.level1_models = [
            lgb.LGBMClassifier(**{**lgb_params_enhanced, 'objective': 'binary', 'metric': 'binary_logloss'}),
            xgb.XGBClassifier(**{**xgb_params_enhanced, 'eval_metric': 'logloss'}),
            CatBoostClassifier(**{**catboost_params, 'loss_function': 'Logloss'}),
            RandomForestClassifier(**{**rf_params, 'class_weight': 'balanced'}),
            
            # Add models optimized for recall/precision balance
            lgb.LGBMClassifier(**{**lgb_params_enhanced, 'scale_pos_weight': 3.5}),
            xgb.XGBClassifier(**{**xgb_params_enhanced, 'scale_pos_weight': 3.5}),
        ]
        
        # Level 2: Meta models with class balancing
        self.level2_models = [
            LogisticRegression(class_weight='balanced'),
            RandomForestClassifier(n_estimators=100, class_weight='balanced'),
            SVC(probability=True, class_weight='balanced', kernel='rbf'),
            lgb.LGBMClassifier(n_estimators=100, scale_pos_weight=2.99)
        ]
        
        # Level 3: Final meta model optimized for F1
        self.final_model = LogisticRegression(class_weight='balanced')
        self.optimal_threshold = 0.5
        
    def fit(self, X, y):
        print("Training Level 1 models...")
        level1_pred = self._get_oof_predictions(X, y, self.level1_models)
        
        print("Training Level 2 models...")
        level2_pred = self._get_oof_predictions(level1_pred, y, self.level2_models)
        
        print("Training final model...")
        self.final_model.fit(level2_pred, y)
        
        # Find optimal threshold using cross-validation
        self.optimal_threshold = self._find_optimal_threshold_cv(level2_pred, y)
        
        # Fit all models on full data
        for model in self.level1_models:
            model.fit(X, y)
        for model in self.level2_models:
            model.fit(level1_pred, y)
            
        return self
    
    def _find_optimal_threshold_cv(self, X, y, cv_folds=3):
        """Find optimal threshold using cross-validation"""
        kf = KFold(n_splits=cv_folds, shuffle=True, random_state=42)
        thresholds = []
        
        for train_idx, val_idx in kf.split(X):
            X_train_fold, X_val_fold = X[train_idx], X[val_idx]
            y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
            
            fold_model = clone(self.final_model)
            fold_model.fit(X_train_fold, y_train_fold)
            y_proba_fold = fold_model.predict_proba(X_val_fold)[:, 1]
            
            threshold, _ = find_optimal_threshold(y_val_fold, y_proba_fold)
            thresholds.append(threshold)
        
        return np.mean(thresholds)
    
    def predict(self, X):
        level1_pred = self._get_test_predictions(X, self.level1_models)
        level2_pred = self._get_test_predictions(level1_pred, self.level2_models)
        y_proba = self.final_model.predict_proba(level2_pred)[:, 1]
        return (y_proba >= self.optimal_threshold).astype(int)
    
    def predict_proba(self, X):
        level1_pred = self._get_test_predictions(X, self.level1_models)
        level2_pred = self._get_test_predictions(level1_pred, self.level2_models)
        return self.final_model.predict_proba(level2_pred)[:, 1]
    
    def _get_test_predictions(self, X, models):
        predictions = np.zeros((X.shape[0], len(models)))
        for i, model in enumerate(models):
            predictions[:, i] = model.predict_proba(X)[:, 1]
        return predictions
    
    def _get_oof_predictions(self, X, y, models):
        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        oof_pred = np.zeros((X.shape[0], len(models)))
        
        for i, model in enumerate(models):
            for train_idx, val_idx in kf.split(X):
                if isinstance(X, pd.DataFrame):
                    X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
                else:
                    X_train_fold, X_val_fold = X[train_idx], X[val_idx]
                y_train_fold = y.iloc[train_idx] if isinstance(y, pd.Series) else y[train_idx]
                
                fold_model = clone(model)
                fold_model.fit(X_train_fold, y_train_fold)
                oof_pred[val_idx, i] = fold_model.predict_proba(X_val_fold)[:, 1]
                
        return oof_pred
    
multi_level_stacking = EnhancedMultiLevelStacking()
multi_level_stacking.fit(X_train, y_train)

y_pred_mls = multi_level_stacking.final_model.predict(
    multi_level_stacking._get_oof_predictions(
        multi_level_stacking._get_oof_predictions(X_val, y_val, multi_level_stacking.level1_models),
        y_val,
        multi_level_stacking.level2_models
    )
)
y_proba_mls = multi_level_stacking.final_model.predict_proba(
    multi_level_stacking._get_oof_predictions(
        multi_level_stacking._get_oof_predictions(X_val, y_val, multi_level_stacking.level1_models),
        y_val,
        multi_level_stacking.level2_models
    )
)[:, 1]

mls_acc = accuracy_score(y_val, y_pred_mls)
mls_auc = roc_auc_score(y_val, y_proba_mls)
mls_f1 = f1_score(y_val, y_pred_mls)
mls_custom_score = custom_score(y_val, y_pred_mls, y_proba_mls)

print(f"\nMulti-level Stacking Classifier Metrics:")
print(f"  Accuracy: {mls_acc:.4f}")
print(f"  ROC-AUC: {mls_auc:.4f}")
print(f"  F1-Score: {mls_f1:.4f}")
print(f"  Custom Score: {mls_custom_score:.4f}")
results['MultiLevelStacking'] = {
    'model': multi_level_stacking,
    'val_auc': mls_auc,
    'val_f1': mls_f1,
    'predictions': y_pred_mls,
    'probabilities': y_proba_mls,
    'custom_score': mls_custom_score
}


--------------------------------------------------------------------------------
4. Multi-level STACKING CLASSIFIER
--------------------------------------------------------------------------------
Training Level 1 models...
Training Level 2 models...
Training final model...

Multi-level Stacking Classifier Metrics:
  Accuracy: 0.7317
  ROC-AUC: 0.8095
  F1-Score: 0.5841
  Custom Score: 0.6875


In [17]:

# ============================================================================
# MODEL COMPARISON AND SELECTION
# ============================================================================
print("\n" + "=" * 80)
print("MODEL COMPARISON SUMMARY")
print("=" * 80)

# Create comparison dataframe
comparison_data = []
for model_name, model_results in results.items():
    if 'cv_scores' in model_results:
        cv_mean = model_results['cv_scores'].mean()
        cv_std = model_results['cv_scores'].std()
    else:
        cv_mean = cv_std = np.nan
    
    comparison_data.append({
        'Model': model_name,
        'CV_AUC_Mean': cv_mean,
        'CV_AUC_Std': cv_std,
        'Val_AUC': model_results['val_auc'],
        'Val_F1': model_results['val_f1'],
        'Custom_Score': model_results['custom_score']
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('Custom_Score', ascending=False)

print("\nModel Performance Comparison:")
print("=" * 60)
for _, row in comparison_df.iterrows():
    # if pd.isna(row['CV_AUC_Mean']):
    #     print(f"{row['Model']:20s} | Val AUC: {row['Val_AUC']:.4f} | Val F1: {row['Val_F1']:.4f}")
    # else:
    #     print(f"{row['Model']:20s} | CV AUC: {row['CV_AUC_Mean']:.4f}±{row['CV_AUC_Std']:.4f} | "
    #           f"Val AUC: {row['Val_AUC']:.4f} | Val F1: {row['Val_F1']:.4f}")
    print(f"{row['Model']:20s} | Val AUC: {row['Val_AUC']:.4f} | Val F1: {row['Val_F1']:.4f} | "
          f"Custom Score: {row['Custom_Score']:.4f}")

# Select best model
best_model_name = comparison_df.iloc[0]['Model']
best_model_results = results[best_model_name]
# best_model_name = 'VotingClassifier'
# best_model_results = results['VotingClassifier']


print(f"\n🏆 BEST MODEL: {best_model_name}")
print(f"   Validation AUC: {best_model_results['val_auc']:.4f}")
print(f"   Validation F1:  {best_model_results['val_f1']:.4f}")
print(f"   Custom Score:   {best_model_results['custom_score']:.4f}")

# ============================================================================
# FEATURE IMPORTANCE ANALYSIS
# ============================================================================
print("\n" + "=" * 80)
print("FEATURE IMPORTANCE ANALYSIS")
print("=" * 80)

def get_feature_importance(model, feature_names, model_type):
    """Extract feature importance from different model types"""
    if hasattr(model, 'feature_importances_'):
        importance = model.feature_importances_
    elif hasattr(model, 'coef_'):
        importance = abs(model.coef_[0])
    else:
        return None
    
    feature_importance = pd.DataFrame({
        'feature': feature_names,
        'importance': importance
    }).sort_values('importance', ascending=False)
    
    return feature_importance

# Get feature importance from best tree-based models
models_for_importance = ['LightGBM', 'XGBoost', 'CatBoost', 'RandomForest']

print("Top 20 Most Important Features:")
print("-" * 50)

for model_name in models_for_importance:
    if model_name in results and 'model' in results[model_name]:
        importance_df = get_feature_importance(
            results[model_name]['model'], 
            X.columns, 
            model_name
        )
        
        if importance_df is not None:
            print(f"\n{model_name}:")
            top_features = importance_df.head(10)
            for idx, row in top_features.iterrows():
                original_name = feature_name_mapping.get(row['feature'], row['feature'])
                print(f"  {row['feature'][:30]:30s} | {row['importance']:.4f}")

# ============================================================================
# SAVE MODELS AND RESULTS
# ============================================================================
print("\n" + "=" * 80)
print("SAVING MODELS AND RESULTS")
print("=" * 80)

import pickle
import os

# Create models directory
os.makedirs('models', exist_ok=True)

# Save best model
best_model = best_model_results['model'] if 'model' in best_model_results else None
if best_model is not None:
    with open(f'models/best_model_{best_model_name.lower()}.pkl', 'wb') as f:
        pickle.dump(best_model, f)
    print(f"✅ Best model ({best_model_name}) saved to models/")

# Save feature columns and preprocessing objects
with open('models/feature_columns.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)

with open('models/feature_name_mapping.pkl', 'wb') as f:
    pickle.dump(feature_name_mapping, f)

with open('models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save results summary
results_summary = {
    'model_comparison': comparison_df.to_dict('records'),
    'best_model_name': best_model_name,
    'best_model_auc': best_model_results['val_auc'],
    'best_model_f1': best_model_results['val_f1'],
    'custom_score': best_model_results['custom_score'],
    'feature_count': len(feature_cols),
    'training_samples': len(X_train),
    'validation_samples': len(X_val)
}

with open('models/training_results.pkl', 'wb') as f:
    pickle.dump(results_summary, f)

print(f"✅ Feature preprocessing objects saved")
print(f"✅ Training results summary saved")

print("\n" + "=" * 80)
print("TRAINING COMPLETED SUCCESSFULLY!")
print("=" * 80)
print(f"🎯 Best Model: {best_model_name}")
print(f"📊 Validation AUC: {best_model_results['val_auc']:.4f}")
print(f"📈 Validation F1: {best_model_results['val_f1']:.4f}")
print(f"🧮 Total Features Used: {len(feature_cols)}")
print(f"💾 Models saved to 'models/' directory")
print("=" * 80)


MODEL COMPARISON SUMMARY

Model Performance Comparison:
VotingClassifier     | Val AUC: 0.8411 | Val F1: 0.6097 | Custom Score: 0.7423
XGBoost              | Val AUC: 0.8376 | Val F1: 0.6201 | Custom Score: 0.7370
F1OptimizedEnsemble  | Val AUC: 0.7666 | Val F1: 0.6271 | Custom Score: 0.7336
stackingClassifier   | Val AUC: 0.8408 | Val F1: 0.6242 | Custom Score: 0.7248
CatBoost             | Val AUC: 0.8369 | Val F1: 0.6197 | Custom Score: 0.7189
LightGBM             | Val AUC: 0.8324 | Val F1: 0.5330 | Custom Score: 0.7170
FocalLossLGBM        | Val AUC: 0.8316 | Val F1: 0.6113 | Custom Score: 0.7122
WeightedEnsemble     | Val AUC: 0.8404 | Val F1: 0.4832 | Custom Score: 0.7031
MultiLevelStacking   | Val AUC: 0.8095 | Val F1: 0.5841 | Custom Score: 0.6875
RandomForest         | Val AUC: 0.8309 | Val F1: 0.1745 | Custom Score: 0.5877

🏆 BEST MODEL: VotingClassifier
   Validation AUC: 0.8411
   Validation F1:  0.6097
   Custom Score:   0.7423

FEATURE IMPORTANCE ANALYSIS
Top 20 Most Im

In [18]:

# ============================================================================
# LOAD AND PREDICT TEST DATA
# ============================================================================

print("\n" + "=" * 80)
print("LOADING TEST DATA")
print("=" * 80)

df_test = pd.read_csv('./testB.csv')
print(f"Test data shape: {df_test.shape}")

# Store IDs
test_ids = df_test['id'].copy()

# Create features
print("\nCreating features for test data...")
df_test_processed = create_advanced_features(df_test, is_train=False)

# Prepare test features
X_test = df_test_processed[feature_cols].copy()
X_test.columns = cleaned_feature_cols

# Fill missing values
print("Handling missing values in test data...")
for col in X_test.columns:
    if col in X.columns:
        if X_test[col].dtype in ['float64', 'int64']:
            X_test[col].fillna(X[col].median(), inplace=True)
        else:
            X_test[col].fillna(X[col].mode()[0] if len(X[col].mode()) > 0 else 0, inplace=True)
    else:
        X_test[col].fillna(0, inplace=True)

# Replace infinities
X_test.replace([np.inf, -np.inf], 0, inplace=True)

print(f"Test feature matrix shape: {X_test.shape}")

# ============================================================================
# MAKE PREDICTIONS
# ============================================================================

print("\n" + "=" * 80)
print("MAKING PREDICTIONS")
print("=" * 80)

predictions = best_model.predict(X_test)
predictions_proba = best_model.predict_proba(X_test)[:, 1]

print(f"Predictions distribution:")
print(f"Class 0: {(predictions == 0).sum()} ({(predictions == 0).sum() / len(predictions) * 100:.1f}%)")
print(f"Class 1: {(predictions == 1).sum()} ({(predictions == 1).sum() / len(predictions) * 100:.1f}%)")

# ============================================================================
# SAVE RESULTS
# ============================================================================

print("\n" + "=" * 80)
print("SAVING RESULTS")
print("=" * 80)

# Create submission dataframe
submission = pd.DataFrame({
    'id': test_ids,
    'label': predictions
})

# Save to CSV
output_path = './submit.csv'
submission.to_csv(output_path, index=False)

print(f"Submission saved to: {output_path}")
print(f"Submission shape: {submission.shape}")
print(f"\nFirst 10 predictions:")
print(submission.head(10))

# Also save with probabilities
submission_with_proba = pd.DataFrame({
    'id': test_ids,
    'label': predictions,
    'probability': predictions_proba
})

output_path_proba = './submit_with_probabilities.csv'
submission_with_proba.to_csv(output_path_proba, index=False)
print(f"\nPredictions with probabilities saved to: {output_path_proba}")

print("\n" + "=" * 80)
print("PIPELINE COMPLETE!")
print("=" * 80)
print(f"Best Model: {best_model_name}")
# print(f"Validation ROC-AUC: {best_auc:.4f}")
print(f"Total Features Used: {len(feature_cols)}")


LOADING TEST DATA
Test data shape: (19999, 87)

Creating features for test data...
Handling missing values in test data...
Test feature matrix shape: (19999, 73)

MAKING PREDICTIONS
Predictions distribution:
Class 0: 15714 (78.6%)
Class 1: 4285 (21.4%)

SAVING RESULTS
Submission saved to: ./submit.csv
Submission shape: (19999, 2)

First 10 predictions:
                                     id  label
0  256b8b06-2fdb-4d63-8571-8db9babddce9      0
1  6de06e34-05cf-4e41-a583-9f26fb0b01a3      0
2  83f213ef-cf80-4a69-956f-c3c0db5316ae      0
3  131f35e1-32ae-48e4-aca9-f32503d21dc7      1
4  ddcc5e03-e2bb-4b87-8132-5383c63f7386      0
5  31d70b86-b3ad-4e28-a6fa-013efdc40cbb      0
6  b60342e1-f1d9-492d-a7cc-f117b1e66cbe      0
7  c4a0e270-cfb1-4aab-b430-46dc555345b3      0
8  96cd2a19-2a73-495c-9b9e-cf5c7a29d2d1      0
9  c0d9e9a3-0daf-4122-9969-e1d467ca5211      0

Predictions with probabilities saved to: ./submit_with_probabilities.csv

PIPELINE COMPLETE!
Best Model: VotingClassifier
Tota